In [1]:
import cv2
import numpy as np
import os

def calculate_abcd(image_path, mask_path):

    image = cv2.imread(image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    img_height, img_width = image.shape[:2]

    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)

    # A asymetry
    # x, y, w, h = cv2.boundingRect(cnt)
    # roi_mask = mask[y:y+h, x:x+w]
    
    # flip_h = cv2.flip(roi_mask, 1)
    # flip_v = cv2.flip(roi_mask, 0)
    
    # diff_h = cv2.bitwise_xor(roi_mask, flip_h)
    # diff_v = cv2.bitwise_xor(roi_mask, flip_v)
    
    # asymmetry_h = np.count_nonzero(diff_h) / area
    # asymmetry_v = np.count_nonzero(diff_v) / area
    # asymmetry_score = (asymmetry_h + asymmetry_v) / 2.0

    # B border
    perimeter = cv2.arcLength(cnt, True)
    border_score = (perimeter ** 2) / (4 * np.pi * area)

    # C color
    _, stddev = cv2.meanStdDev(image, mask=mask)
    color_score = float(np.sum(stddev) / (3.0 * 128.0)) # do poprawy

    # D diameter
    #_, radius = cv2.minEnclosingCircle(cnt)
    #diameter_pixels = radius * 2
    if len(cnt) >= 5:
        rect = cv2.fitEllipse(cnt)
        (x_e, y_e) = rect[0]
        (w_e, h_e) = rect[1]
        angle = rect[2]

        if w_e < h_e:
            if angle < 90:
                angle -= 90
            else:
                angle += 90
                
        rows, cols = mask.shape
        rot = cv2.getRotationMatrix2D((x_e, y_e), angle, 1)
        cos_ang = np.abs(rot[0, 0])
        sin_ang = np.abs(rot[0, 1])
        
        nW_rot = int((rows * sin_ang) + (cols * cos_ang))
        nH_rot = int((rows * cos_ang) + (cols * sin_ang))

        rot[0, 2] += (nW_rot / 2) - cols / 2
        rot[1, 2] += (nH_rot / 2) - rows / 2

        warp_mask = cv2.warpAffine(mask, rot, (nW_rot, nH_rot))

        cnts_warp, _ = cv2.findContours(warp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts_warp:
            contour_warp = max(cnts_warp, key=cv2.contourArea)
            xx, yy, nW, nH = cv2.boundingRect(contour_warp)
            warp_mask_cropped = warp_mask[yy:yy + nH, xx:xx + nW]
        else:
            nW, nH = w_e, h_e
            warp_mask_cropped = warp_mask
    else:
        xx, yy, nW, nH = cv2.boundingRect(cnt)
        warp_mask_cropped = mask[yy:yy+nH, xx:xx+nW]

    major_axis_px = max([nW, nH])
    #minor_axis_px = min([nW, nH])

    img_diagonal = np.sqrt(img_width**2 + img_height**2)
    diameter_score = major_axis_px / img_diagonal

    # A remake

    flip_h = cv2.flip(warp_mask_cropped, 1)
    flip_v = cv2.flip(warp_mask_cropped, 0)
    
    diff_h = cv2.bitwise_xor(warp_mask_cropped, flip_h)
    diff_v = cv2.bitwise_xor(warp_mask_cropped, flip_v)
    
    asymmetry_h = np.count_nonzero(diff_h) / area
    asymmetry_v = np.count_nonzero(diff_v) / area
    asymmetry_score = (asymmetry_h + asymmetry_v) / 2.0

    return {
        "A_Asymmetry": round(asymmetry_score, 2),
        "B_Border": round(border_score, 2),
        "C_Color": round(color_score, 2),
        "D_Diameter": round(diameter_score, 2)
    }

if __name__ == "__main__":
    img_path = "ISIC_0000000.jpg"
    mask_path = "ISIC2018_Task1_Training_GroundTruth/ISIC_0000000_segmentation.png"
    
    if os.path.exists(img_path) and os.path.exists(mask_path):
        features = calculate_abcd(img_path, mask_path)
        for key, value in features.items():
            print(f"- {key}: {value}")

In [ ]:
# imgSrc="dataset/test/testInput/ISIC_0000000.jpg"
# maskSrc="dataset/test/testGt/ISIC_0000000_segmentation.png"

# result = calculate_abcd(imgSrc, maskSrc)

In [ ]:
# print(result["A_Asymmetry"])
# print(result["B_Border"])
# print(result["C_Color"])
# print(result["D_Diameter"])

0.18
1.67
0.31
0.68


In [ ]:
from pathlib import Path
import csv

input_dir=Path("SegDataset/val/valInput")
mask_dir=Path("SegDataset/val/valGt")

csv_path=Path("ValLesionParameters.csv")
file_exists = csv_path.exists()

with open(csv_path, mode='a', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    
    if not file_exists:
        writer.writerow(['image', 'A', 'B', 'C', 'D'])

    for input_path in input_dir.glob("*.jpg"):
        
        mask_filename = f"{input_path.stem}_segmentation.png"
        
        mask_path = mask_dir / mask_filename
        
        if mask_path.exists():
            result = calculate_abcd(input_path, mask_path)
            A = result["A_Asymmetry"]
            B = result["B_Border"]
            C = result["C_Color"]
            D = result["D_Diameter"]
            writer.writerow([input_path.name, A, B, C, D])
        else:
            print(f"OSTRZEŻENIE: Nie znaleziono maski {mask_filename} dla obrazu {input_path.name}")